# [SOLUTION] Udaplay Project

## Part 02 - Agent (Solution)

This notebook implements a working version of the UdaPlay agent with three tools:
- `retrieve_game` (local RAG retrieval)
- `evaluate_retrieval` (assess retrieved results)
- `game_web_search` (fallback to web search via Tavily)

The agent maintains conversation state, calls tools as needed, and provides structured answers with citations.

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import BaseMessage
from lib.tooling import tool

# Via Tavily client for web search
from tavily import TavilyClient

In [3]:
# Load environment variables
# Explicitly pass dotenv_path to handle VS Code local environments where the
# kernel cwd may be the workspace root rather than this notebook's folder.
from pathlib import Path
_env_path = next(
    (p for p in [Path('.env'), Path('project/starter/.env')] if p.exists()),
    Path('.env')
)
load_dotenv(dotenv_path=_env_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")

assert OPENAI_API_KEY, "OPENAI_API_KEY is required"
assert CHROMA_OPENAI_API_KEY, "CHROMA_OPENAI_API_KEY is required"
assert TAVILY_API_KEY, "TAVILY_API_KEY is required"

import openai
openai.api_key = OPENAI_API_KEY
openai.base_url = OPENAI_API_BASE or openai.base_url

In [4]:
# Instantiate ChromaDB client and get the collection used in Part 1
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    api_base=OPENAI_API_BASE
)

collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

print(f"Loaded collection '{collection.name}' — {collection.count()} documents available for retrieval.")

Loaded collection 'udaplay' — 25 documents available for retrieval.


### Tools

The agent uses three tools:

- **retrieve_game**: Searches the local vector database (ChromaDB)
- **evaluate_retrieval**: Scores the usefulness of the retrieved documents
- **game_web_search**: Fallback search using Tavily

In [5]:
@tool
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """Search the vector database to retrieve game info relevant to the query."""

    # Normalize query: the LLM may pass a non-string type (list, dict, None)
    # as the query argument. Convert to string and bail out if empty.
    if not isinstance(query, str):
        query = str(query)
    query = query.strip()
    if not query:
        return []

    results = collection.query(query_texts=[query], n_results=n_results)

    docs = []
    for i, doc in enumerate(results.get("documents", [[]])[0]):
        metadata = results.get("metadatas", [[]])[0][i] if results.get("metadatas") else {}
        doc_id = results.get("ids", [[]])[0][i] if results.get("ids") else None
        docs.append({
            "id": doc_id,
            "content": doc,
            "metadata": metadata,
        })

    return docs

In [6]:
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser


class RetrievalEvaluation(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents provide enough information to answer the question")
    description: str = Field(description="A short explanation of the evaluation decision")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> dict:
    """Evaluate whether the retrieved documents are sufficient to answer the user's question."""

    # The agent may pass retrieved_docs as a JSON string (from tool call serialization),
    # or the list elements may be plain strings rather than dicts.
    if isinstance(retrieved_docs, str):
        try:
            retrieved_docs = json.loads(retrieved_docs)
        except Exception:
            pass

    if isinstance(retrieved_docs, dict):
        retrieved_docs = [retrieved_docs]

    if not isinstance(retrieved_docs, list):
        retrieved_docs = [retrieved_docs] if retrieved_docs else []

    if not retrieved_docs:
        return {
            "useful": False,
            "description": "No documents were retrieved from the local knowledge base.",
        }

    # Build a readable context string from the retrieved documents
    context_parts = []
    for doc in retrieved_docs:
        if isinstance(doc, dict):
            context_parts.append(doc.get("content", str(doc)))
        else:
            context_parts.append(str(doc))
    context = "\n\n".join(context_parts)

    # Use the LLM as a strict judge — Module 10 pattern:
    #   LLM.invoke(input, response_format=PydanticModel) → AIMessage
    #   PydanticOutputParser.parse(ai_message)           → typed result
    # This replaces the naive token-matching heuristic with a proper semantic
    # check: if the question names a specific game not present in the docs,
    # the judge returns useful=False and the agent falls back to web search.
    judge_prompt = (
        "You are a strict retrieval-quality evaluator. "
        "Decide whether the provided documents contain enough information "
        "to directly and accurately answer the user's question. "
        "Be strict: if the question asks about a specific game or entity "
        "that is NOT explicitly mentioned in the documents, set useful=False.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{context}\n\n"
        "Do these documents contain sufficient information to answer the question?"
    )

    llm_judge = LLM(model="gpt-4o-mini")
    ai_message = llm_judge.invoke(input=judge_prompt, response_format=RetrievalEvaluation)

    parser = PydanticOutputParser(model_class=RetrievalEvaluation)
    try:
        evaluation = parser.parse(ai_message)
        return {"useful": evaluation.useful, "description": evaluation.description}
    except Exception as e:
        # Fallback: if parsing fails, default to web search so we never silently drop data
        return {
            "useful": False,
            "description": f"Evaluation parsing failed ({e}); defaulting to web search fallback.",
        }

In [7]:
@tool
def game_web_search(question: str, max_results: int = 3) -> dict:
    """Search the web using Tavily and return a summarized response."""

    client = TavilyClient(api_key=TAVILY_API_KEY)
    search_result = client.search(
        query=question,
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    return {
        "answer": search_result.get("answer"),
        "results": search_result.get("results", [])[:max_results],
        "search_metadata": {
            "query": question,
            "timestamp": __import__("datetime").datetime.now().isoformat(),
        },
    }

### Agent

In [8]:
instructions = (
    "You are UdaPlay, an AI research assistant specialized in video games. "
    "Use the provided tools to answer the user's question. "
    "First, attempt to retrieve information using `retrieve_game`. "
    "Then, use `evaluate_retrieval` to decide whether the retrieved information is sufficient. "
    "If the local knowledge is not sufficient, use `game_web_search` to look up information online. "
    "You MUST append a citation at the end of every answer — no exceptions. "
    "If the answer came from the local database, end with: [Source: Internal Database]. "
    "If the answer came from a web search, end with the exact URL(s) from the search results, "
    "formatted as: [Source: <url>]. "
    "If both sources were used, cite both. Never omit a citation."
)

tools = [retrieve_game, evaluate_retrieval, game_web_search]

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=tools,
)

print("Agent initialized with tools:", [t.name for t in tools])

Agent initialized with tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']


### Reporting

The `print_messages` helper prints the full message trace for each agent run — including the agent's reasoning (tool call decisions) and tool usage results — followed by the final answer. This satisfies the rubric requirement that output shows the agent's reasoning, tool usage, and final answer.

In [9]:
from lib.messages import BaseMessage, AIMessage, UserMessage, ToolMessage, SystemMessage

def print_messages(messages: list[BaseMessage]):
    """Print the agent message trace with clear labels for each role.

    Distinguishes system instructions, user input, tool-call decisions
    (with arguments), tool results, and the final assistant answer —
    giving reviewers a step-by-step audit trail of the agent's reasoning.
    """
    step = 0
    for m in messages:
        if m.role == "system":
            # Summarise rather than dump the full system prompt
            print(f"  [SYSTEM ] Instructions loaded ({len(m.content or '')} chars)")

        elif m.role == "user":
            print(f"  [USER   ] {m.content}")

        elif m.role == "assistant":
            tool_calls = getattr(m, "tool_calls", None)
            if tool_calls:
                for tc in tool_calls:
                    step += 1
                    try:
                        args = json.loads(tc.function.arguments)
                        args_str = json.dumps(args)
                    except Exception:
                        args_str = tc.function.arguments
                    print(f"  [STEP {step:>2}] Tool call  →  {tc.function.name}({args_str})")
            elif m.content:
                print(f"  [ANSWER ] {m.content}")

        elif m.role == "tool":
            name = getattr(m, "name", "tool")
            try:
                # content is json.dumps(str(result)) — double-encoded
                raw = json.loads(m.content)
                if isinstance(raw, str):
                    raw = json.loads(raw)
                preview = json.dumps(raw)
            except Exception:
                preview = str(m.content)
            # Truncate long results for readability
            truncated = (preview[:300] + "  …[truncated]") if len(preview) > 300 else preview
            print(f"  [RESULT ] {name}  →  {truncated}")

In [10]:
queries = [
    # Q1: Theme query — answered from internal DB (fantasy/dragon games exist in the 25-game dataset)
    "Which games in your database are set in a fantasy world and involve dragons?",

    # Q2: Context-dependent follow-up — proves short-term memory (agent must recall Q1's results)
    "What platforms were those games released on?",

    # Q3: Named game NOT in the 25-game local DB — evaluate_retrieval will flag results as not useful,
    #     forcing the agent to fall back to game_web_search and cite a live URL.
    "What year was Elden Ring released, and which platforms is it available on?",

    # Q4: Broad recommendation — answered from internal DB
    "Recommend games that are immersive and have strong storytelling elements.",
]

previous_message_count = 0
for query in queries:
    print("=" * 62)
    print(f"Query: {query}")
    print()
    run = agent.invoke(query)
    final_state = run.get_final_state()
    messages = final_state["messages"]

    # Slice to only the messages added during this invocation
    new_messages = messages[previous_message_count:]
    previous_message_count = len(messages)

    # Count tool calls in this turn for the summary line
    tool_call_count = sum(
        len(m.tool_calls) for m in new_messages
        if m.role == "assistant" and getattr(m, "tool_calls", None)
    )

    print("Agent reasoning & tool usage:")
    print_messages(new_messages)
    print()
    print(f"Summary: {tool_call_count} tool call(s) made this turn.")
    print()
    print("Final answer:")
    print(messages[-1].content)
    print()

print("=" * 62)
print(f"All {len(queries)} queries completed successfully.")

Query: Which games in your database are set in a fantasy world and involve dragons?

Agent reasoning & tool usage:
  [SYSTEM ] Instructions loaded (705 chars)
  [USER   ] Which games in your database are set in a fantasy world and involve dragons?
  [STEP  1] Tool call  →  retrieve_game({"query": "fantasy world dragons"})
  [RESULT ] retrieve_game  →  "[{'id': '025', 'content': '[PC] Neverwinter Nights (2002) - A 3D role-playing game developed by BioWare based on Dungeons & Dragons 3rd Edition rules, set in the Forgotten Realms city of Neverwinter where a deadly plague threatens the population. The game featured robust multiplayer and a powerful   …[truncated]
  [STEP  2] Tool call  →  evaluate_retrieval({"question": "Which games in your database are set in a fantasy world and involve dragons?", "retrieved_docs": ["[PC] Neverwinter Nights (2002) - A 3D role-playing game developed by BioWare based on Dungeons & Dragons 3rd Edition rules, set in the Forgotten Realms city of Neverwinter w

### (Optional) Advanced

In [11]:
# For extra credit, you can extend the agent to include:
# - Long-term memory (persisting new facts from the web)
# - A more advanced state machine that explicitly branches between tools
# - Structured output (JSON) for downstream consumption